# FiLM-GraspNet A1 Training on Kaggle

Before running: choose **GPU T4 x2** and turn **Internet on** in Kaggle settings. Add a Kaggle Dataset containing `train_pilot.tar.gz`, `val_pilot.tar.gz`, `grconvnet3_pretrained.pth`, and `SHA256SUMS.txt`. This notebook uses only one T4. Replace the dataset name in Cell 3. The A1 code must be available in the GitHub repository before training.


# System Checking


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Set before importing torch

import sys
import torch
import numpy as np

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "Enable GPU in Kaggle settings"
assert torch.cuda.device_count() == 1, "Restart the kernel and run this cell first"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
assert "T4" in torch.cuda.get_device_name(0), "A T4 is required for this run"


In [ ]:
from pathlib import Path
import subprocess

TRANSFER = Path("/kaggle/input/REPLACE_WITH_YOUR_DATASET_NAME")
for name in ("train_pilot.tar.gz", "val_pilot.tar.gz", "grconvnet3_pretrained.pth", "SHA256SUMS.txt"):
    assert (TRANSFER / name).is_file(), f"Missing: {TRANSFER / name}"

subprocess.run(["sha256sum", "-c", "SHA256SUMS.txt"], cwd=TRANSFER, check=True)
print("KAGGLE INPUT CHECK PASSED")


# Git Clone


In [ ]:
%cd /kaggle/working
!git clone https://github.com/nammannam/film-graspnet.git
%cd /kaggle/working/film-graspnet
!git rev-parse HEAD


# Dependencies installation


In [ ]:
%pip install -q ftfy regex tqdm tensorboard scipy scikit-image
%pip install -q --no-deps git+https://github.com/openai/CLIP.git
%pip install -q --no-deps -e /kaggle/working/film-graspnet

import clip
import filmgrasp
print("CLIP and FiLM-GraspNet imports: PASSED")


# Dataset


In [ ]:
DATA = Path("/kaggle/working/data")
DATA.mkdir(parents=True, exist_ok=True)

for name in ("train_pilot", "val_pilot"):
    subprocess.run(["tar", "-xzf", str(TRANSFER / f"{name}.tar.gz"), "-C", str(DATA)], check=True)
    assert (DATA / name / "manifest.json").is_file()
    print(name, "EXTRACTED")


In [ ]:
import hashlib
from filmgrasp.data.dataset import GraspAnythingPPDataset

EXPECTED_MANIFESTS = {
    "train_pilot": "7038dc3929deae9a729ed75d854e241f70729f41d22fea2b67520374c96225ff",
    "val_pilot": "4f93a67ab3d636489f784ca6d9c42de0ef99f7de0766ac1890cb1870fb31f7cc",
}
for name, expected in EXPECTED_MANIFESTS.items():
    actual = hashlib.sha256((DATA / name / "manifest.json").read_bytes()).hexdigest()
    assert actual == expected, f"{name} manifest differs from B0: {actual}"

pretrained = TRANSFER / "grconvnet3_pretrained.pth"
assert hashlib.sha256(pretrained.read_bytes()).hexdigest() == (
    "2fdb9867125eb7cbe5f05b0ee1c83fad29f157e39575b69da10868faf71349a4"
)
train_ds = GraspAnythingPPDataset(str(DATA / "train_pilot"))
val_ds = GraspAnythingPPDataset(str(DATA / "val_pilot"))
assert len(train_ds) == 381 and len(val_ds) == 76
print("DATASET CHECK PASSED:", len(train_ds), "train /", len(val_ds), "validation")


# Training


In [ ]:
%cd /kaggle/working/film-graspnet

from filmgrasp.training.train import parse_args
assert parse_args(["--dropout-prob", "0.1"]).dropout_prob == 0.1

RUN_NAME = "filmgrasp_kaggle_a1_dropout_p01_seed42_100e"
RUN = Path("/kaggle/working/runs") / RUN_NAME
assert not RUN.exists(), f"Run already exists: {RUN}"
print("A1 CLI and new-run check: PASSED")


In [ ]:
%cd /kaggle/working/film-graspnet

!python -m filmgrasp.training.train \
  --train-root /kaggle/working/data/train_pilot \
  --val-root /kaggle/working/data/val_pilot \
  --epochs 100 \
  --batch-size 4 \
  --lr 1e-4 \
  --device cuda \
  --num-workers 0 \
  --seed 42 \
  --pretrained {pretrained} \
  --dropout-prob 0.1 \
  --output-root /kaggle/working/runs \
  --run-name filmgrasp_kaggle_a1_dropout_p01_seed42_100e


# Results

The run writes `best.pt`, `last.pt`, `metrics.csv`, `config.json`, `data_summary.json`, and `train.log` under `/kaggle/working/runs/filmgrasp_kaggle_a1_dropout_p01_seed42_100e/`. Use **Save Version** with output files, or download the run before ending the session. For a later resume, attach the saved A1 run as Kaggle input, copy it back to `/kaggle/working/runs/`, and add `--resume .../checkpoints/last.pt` to the same training command. Never resume from B0. Compare A1 and B0 using the same post-training evaluator and validation manifest.


In [ ]:
for name in ("checkpoints/best.pt", "checkpoints/last.pt", "metrics.csv", "config.json", "data_summary.json", "train.log"):
    assert (RUN / name).is_file(), f"Missing: {RUN / name}"
print("TRAINING OUTPUT CHECK PASSED:", RUN)
